In [1]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from src.data.hand_landmarks import HandLandmarksDataset
from src.models.SLT_model import SignLanguageTranslator
import torch
import gc
from tqdm import tqdm
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from config import ROOT
from functools import partial
from transformers import MT5ForConditionalGeneration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_DIR = os.path.join(
    ROOT,
    "datasets",
    "processed",
    "mediapipe"
)
SAVE_DIR = os.path.join(
    ROOT,
    "models"
)
os.makedirs(SAVE_DIR, exist_ok=True)
BATCH_SIZE = 8
EPOCHS = 10
LR = 3e-5

MAX_DATASET_LENGTH = 1000

C:\Users\2507\.conda\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained(
        "google/mt5-small",
        use_fast=False
)
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.parameters()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


<generator object Module.parameters at 0x000002AF9FA6DEB0>

In [21]:
def collate_fn(batch, tokenizer):

    features = []
    texts = []
    text_masks = []

    for feature, text, mask in batch:
        features.append(feature)
        texts.append(text)
        text_masks.append(mask)

    lengths = [feature.shape[0] for feature in features]

    features = pad_sequence(
        features,
        batch_first=True
    )

    texts = pad_sequence(
        texts,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    max_len = features.shape[1]

    video_mask = (
            torch.arange(max_len)[None, :]
            < torch.tensor(lengths)[:, None]
    )

    video_mask = video_mask.bool()

    labels = texts.clone()

    labels[
        labels == tokenizer.pad_token_id
    ] = -100

    return features, labels, video_mask


dataset = HandLandmarksDataset(
        FEATURE_DIR,
        tokenizer,
        MAX_DATASET_LENGTH
    )

total_size = len(dataset)

train_size = int(total_size * 0.8)
val_size = int(total_size * 0.1)
test_size = int(total_size * 0.1)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=partial(
        collate_fn,
        tokenizer=tokenizer
    ),
    pin_memory=True,
    num_workers=4
)

